# Sıfırdan küçük CNN (MTG-Jamendo mood/theme)
Accelerator: **GPU T4 x2**, Internet: **On**.

**Adım A** log-mel çıkarımı (`pipeline.py --models mel`, önce 2 tar'lık deneme, sonra tam koşu, Commit modu). **Adım B** CNN eğitimi (`train_cnn.py`). A bittikten sonra çıktıyı dataset olarak ekle, B'yi ayrı oturumda çalıştır.

In [ ]:
REPO_URL = "https://github.com/muzik-oneri/musicrec.git"
!rm -rf /kaggle/temp/musicrec /kaggle/temp/mtg
!git clone -q $REPO_URL /kaggle/temp/musicrec
!git clone -q https://github.com/MTG/mtg-jamendo-dataset /kaggle/temp/mtg
!pip install -q -r /kaggle/temp/musicrec/requirements.txt
!cd /kaggle/temp/musicrec && git log -1 --format="KOD SÜRÜMÜ: %h %s" && ls src/extract_mel.py src/train_cnn.py

## Adım A: log-mel çıkarımı
Önce deneme. Tar başına süreyi ve `du -sh` ile çıktı boyutunu not et; tam boyut = tar başına boyut x 100.

In [ ]:
!python /kaggle/temp/musicrec/src/pipeline.py \
    --mtg-dir /kaggle/temp/mtg \
    --out-dir /kaggle/working/emb_mel \
    --models mel --max-windows 3 --limit-tars 2
!du -sh /kaggle/working/emb_mel/mel

In [ ]:
# TAM KOŞU (deneme makulse, Save & Run All / Commit ile). Süre dolarsa --prev-dir ile devam.
# !python /kaggle/temp/musicrec/src/pipeline.py \
#     --mtg-dir /kaggle/temp/mtg --out-dir /kaggle/working/emb_mel \
#     --models mel --max-windows 3 --max-hours 10.5
#     # --prev-dir /kaggle/input/musicrec-emb-mel

## Adım B: CNN eğitimi
Önce duman testi (`--epochs 2 --max-train 500 --seeds 1`), sonra tek seed tam koşu, sonra 5 seed.

In [ ]:
EMB = "/kaggle/working/emb_mel"   # B ayrı oturumdaysa: /kaggle/input/<dataset adı>
!python /kaggle/temp/musicrec/src/train_cnn.py --emb-dir $EMB --out-dir /kaggle/working/runs_cnn \
    --epochs 2 --max-train 500 --seeds 1 --pos-weight

In [ ]:
!python /kaggle/temp/musicrec/src/train_cnn.py --emb-dir $EMB --out-dir /kaggle/working/runs_cnn \
    --epochs 40 --seeds 1 --pos-weight

In [ ]:
# 5 seed (tek seed iyi çıktıysa)
# !python /kaggle/temp/musicrec/src/train_cnn.py --emb-dir $EMB --out-dir /kaggle/working/runs_cnn \
#     --epochs 40 --seeds 1 2 3 4 5 --pos-weight